In [0]:
from pyspark.sql import functions as F

In [0]:
# ----------------------------------------------------------------------------------------
#  Step 2 -  Bronze Layer insert raw layer 
# ----------------------------------------------------------------------------------------

#  --- 2A. read order.json (nested/newline- delimited)-----------------------------------
df_orders_raw = (
    spark.read
 .option("inferSchema", "true")               # auto-delete nested structType = ArrayType  
.json("/Volumes/bigdata2/my_voloum/use-case2/ecommerce/orders.json") )

print("\n ------------------------- orders.json schema --------------------------")
df_orders_raw.printSchema()
display(df_orders_raw)

#  ---- expload: flattern nested items array -> one raw per line item ---------------------------------

df_orders_exploded = (
    df_orders_raw
    .withColumn("items",F.explode("items"))  # arrayType -> individual structs
    .select(
        "order_id",
        "customer_id",
        "order_date",
        "order_status",
        "payment_method",
        "shipping_city",
        "discount_pct",
        F.col("items.item_id").alias("item_id"),
        F.col("items.product_name").alias("product_name"),
        F.col("items.category").alias("category"),
        F.col("items.quantity").alias("quantity"),
        F.col("items.unit_price").alias("unit_price"),
        F.col("items.line_total").alias("line_total")
    )
)
display(df_orders_exploded)


 ------------------------- orders.json schema --------------------------
root
 |-- customer_id: string (nullable = true)
 |-- discount_pct: long (nullable = true)
 |-- items: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- category: string (nullable = true)
 |    |    |-- item_id: string (nullable = true)
 |    |    |-- line_total: double (nullable = true)
 |    |    |-- product_name: string (nullable = true)
 |    |    |-- quantity: long (nullable = true)
 |    |    |-- unit_price: double (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- shipping_city: string (nullable = true)



customer_id,discount_pct,items,order_date,order_id,order_status,payment_method,shipping_city
CUST0044,15,"List(List(Clothing, ITEM350, 3720.67, Product_Clothing_1, 1, 3720.67))",2024-02-22,ORD00001,pending,credit_card,Hyderabad
CUST0036,5,"List(List(Sports, ITEM338, 513.79, Product_Sports_1, 1, 513.79))",2024-01-14,ORD00002,shipped,wallet,Chennai
CUST0025,20,"List(List(Clothing, ITEM990, 7135.2, Product_Electronics_1, 5, 1427.04), List(Books, ITEM259, 6936.96, Product_Clothing_2, 4, 1734.24), List(Books, ITEM199, 509.09, Product_Books_3, 1, 509.09), List(Sports, ITEM144, 6796.9, Product_Home_4, 5, 1359.38))",2024-03-04,ORD00003,delivered,wallet,Bangalore
CUST0023,20,"List(List(Clothing, ITEM171, 5009.15, Product_Electronics_1, 5, 1001.83), List(Electronics, ITEM975, 14780.55, Product_Clothing_2, 3, 4926.85), List(Clothing, ITEM750, 5703.88, Product_Books_3, 4, 1425.97))",2024-07-08,ORD00004,shipped,UPI,Mumbai
CUST0021,10,"List(List(Home, ITEM267, 18295.9, Product_Home_1, 5, 3659.18), List(Clothing, ITEM755, 14844.42, Product_Sports_2, 3, 4948.14))",2024-12-16,ORD00005,delivered,debit_card,Mumbai
CUST0015,0,"List(List(Clothing, ITEM680, 1132.89, Product_Books_1, 3, 377.63), List(Clothing, ITEM758, 8033.52, Product_Home_2, 4, 2008.38), List(Sports, ITEM862, 2223.51, Product_Sports_3, 3, 741.17), List(Home, ITEM538, 11243.16, Product_Sports_4, 3, 3747.72))",2024-07-04,ORD00006,shipped,wallet,Chennai
CUST0025,20,"List(List(Sports, ITEM263, 806.56, Product_Home_1, 1, 806.56))",2024-02-02,ORD00007,returned,wallet,Chennai
CUST0033,20,"List(List(Sports, ITEM111, 21554.3, Product_Electronics_1, 5, 4310.86), List(Books, ITEM448, 11563.68, Product_Electronics_2, 3, 3854.56), List(Sports, ITEM103, 3331.52, Product_Books_3, 4, 832.88))",2024-04-01,ORD00008,delivered,UPI,Hyderabad
CUST0020,20,"List(List(Sports, ITEM265, 3801.74, Product_Sports_1, 2, 1900.87), List(Electronics, ITEM600, 3014.78, Product_Electronics_2, 1, 3014.78))",2024-07-04,ORD00009,shipped,credit_card,Delhi
CUST0009,10,"List(List(Sports, ITEM935, 3672.94, Product_Electronics_1, 1, 3672.94))",2024-03-05,ORD00010,returned,wallet,Delhi


order_id,customer_id,order_date,order_status,payment_method,shipping_city,discount_pct,item_id,product_name,category,quantity,unit_price,line_total
ORD00001,CUST0044,2024-02-22,pending,credit_card,Hyderabad,15,ITEM350,Product_Clothing_1,Clothing,1,3720.67,3720.67
ORD00002,CUST0036,2024-01-14,shipped,wallet,Chennai,5,ITEM338,Product_Sports_1,Sports,1,513.79,513.79
ORD00003,CUST0025,2024-03-04,delivered,wallet,Bangalore,20,ITEM990,Product_Electronics_1,Clothing,5,1427.04,7135.2
ORD00003,CUST0025,2024-03-04,delivered,wallet,Bangalore,20,ITEM259,Product_Clothing_2,Books,4,1734.24,6936.96
ORD00003,CUST0025,2024-03-04,delivered,wallet,Bangalore,20,ITEM199,Product_Books_3,Books,1,509.09,509.09
ORD00003,CUST0025,2024-03-04,delivered,wallet,Bangalore,20,ITEM144,Product_Home_4,Sports,5,1359.38,6796.9
ORD00004,CUST0023,2024-07-08,shipped,UPI,Mumbai,20,ITEM171,Product_Electronics_1,Clothing,5,1001.83,5009.15
ORD00004,CUST0023,2024-07-08,shipped,UPI,Mumbai,20,ITEM975,Product_Clothing_2,Electronics,3,4926.85,14780.55
ORD00004,CUST0023,2024-07-08,shipped,UPI,Mumbai,20,ITEM750,Product_Books_3,Clothing,4,1425.97,5703.88
ORD00005,CUST0021,2024-12-16,delivered,debit_card,Mumbai,10,ITEM267,Product_Home_1,Home,5,3659.18,18295.9


In [0]:
from pyspark.sql import functions as F

#  --- Read JSON ---
df_orders_raw = (
    spark.read
        .option("inferSchema", "true")
        .json("/Volumes/bigdata2/my_voloum/use-case2/ecommerce/orders.json")   # ⚠ fixed path
)

print("\n---------------- orders.json schema ----------------")
df_orders_raw.printSchema()


# --- Explode items array ---
df_orders_exploded = (
    df_orders_raw
        .withColumn("item", F.explode("items"))   # explode array
        .select(
            "order_id",
            "customer_id",
            "order_date",
            "order_status",
            "payment_method",
            "shipping_city",
            "discount_pct",
            F.col("item.item_id").alias("item_id"),
            F.col("item.product_name").alias("product_name"),
            F.col("item.category").alias("category"),
            F.col("item.quantity").alias("quantity"),
            F.col("item.unit_price").alias("unit_price"),
            F.col("item.line_total").alias("line_total")
        )
)

display(df_orders_exploded)


---------------- orders.json schema ----------------
root
 |-- customer_id: string (nullable = true)
 |-- discount_pct: long (nullable = true)
 |-- items: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- category: string (nullable = true)
 |    |    |-- item_id: string (nullable = true)
 |    |    |-- line_total: double (nullable = true)
 |    |    |-- product_name: string (nullable = true)
 |    |    |-- quantity: long (nullable = true)
 |    |    |-- unit_price: double (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- shipping_city: string (nullable = true)



order_id,customer_id,order_date,order_status,payment_method,shipping_city,discount_pct,item_id,product_name,category,quantity,unit_price,line_total
ORD00001,CUST0044,2024-02-22,pending,credit_card,Hyderabad,15,ITEM350,Product_Clothing_1,Clothing,1,3720.67,3720.67
ORD00002,CUST0036,2024-01-14,shipped,wallet,Chennai,5,ITEM338,Product_Sports_1,Sports,1,513.79,513.79
ORD00003,CUST0025,2024-03-04,delivered,wallet,Bangalore,20,ITEM990,Product_Electronics_1,Clothing,5,1427.04,7135.2
ORD00003,CUST0025,2024-03-04,delivered,wallet,Bangalore,20,ITEM259,Product_Clothing_2,Books,4,1734.24,6936.96
ORD00003,CUST0025,2024-03-04,delivered,wallet,Bangalore,20,ITEM199,Product_Books_3,Books,1,509.09,509.09
ORD00003,CUST0025,2024-03-04,delivered,wallet,Bangalore,20,ITEM144,Product_Home_4,Sports,5,1359.38,6796.9
ORD00004,CUST0023,2024-07-08,shipped,UPI,Mumbai,20,ITEM171,Product_Electronics_1,Clothing,5,1001.83,5009.15
ORD00004,CUST0023,2024-07-08,shipped,UPI,Mumbai,20,ITEM975,Product_Clothing_2,Electronics,3,4926.85,14780.55
ORD00004,CUST0023,2024-07-08,shipped,UPI,Mumbai,20,ITEM750,Product_Books_3,Clothing,4,1425.97,5703.88
ORD00005,CUST0021,2024-12-16,delivered,debit_card,Mumbai,10,ITEM267,Product_Home_1,Home,5,3659.18,18295.9
